To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding versions!
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.5.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [5]:
from google.colab import files
uploaded = files.upload()

Saving train.jsonl to train.jsonl
Saving test.jsonl to test.jsonl


<a name="Data"></a>
### Data Prep
We now use the [Alpaca dataset](https://huggingface.co/datasets/unsloth/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read our docs [here](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Conversational.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [6]:
# ════════════════════════════════════════════════════════════════════
# DATA PREP — Load OUR dataset (train.jsonl) and apply Qwen's chat template
# Replaces the original Alpaca cell entirely.
# ════════════════════════════════════════════════════════════════════
from datasets import load_dataset

# Load our chat-format dataset (each row has a "messages" list)
dataset = load_dataset("json", data_files="train.jsonl", split="train")

# Apply Qwen's built-in chat template to turn messages -> a single text string
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,           # we want the string, not token IDs
            add_generation_prompt=False,  # full conversation incl. assistant answer
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# Sanity check: how many examples + what one formatted example looks like
print("Number of training examples:", len(dataset))
print("\n--- One formatted example ---\n")
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/680 [00:00<?, ? examples/s]

Number of training examples: 680

--- One formatted example ---

<|im_start|>system
You are a DevOps log classifier. Given a raw log line, output ONLY a valid JSON object with these exact fields:
 
{
  "category": one of ["Network", "Database", "Memory", "Security", "Application", "System"],
  "source": one of ["Docker", "Kubernetes", "Server", "Database", "OS", "Network_Device"],
  "severity": one of ["Warning", "Critical", "Fatal"],
  "summary": a short human-readable summary of the problem (max 15 words)
}
 
FIELD DEFINITIONS:
- "category" = WHAT kind of issue it is:
    • Network    → connections, timeouts, packets, sockets, DNS, ports
    • Database   → data storage, blocks, reads/writes, queries, replication, file system data ops (e.g. HDFS block storage)
    • Memory     → RAM, heap, OOM, leaks, allocation, cache eviction
    • Security   → auth, login attempts, permissions, intrusion, SSH, certificates
    • Application→ app-level logic, services, request handling, normal app o

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [7]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,        # ← train on the FULL dataset 3 times
        # max_steps = 60,            # ← disabled: we use epochs instead
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,         # ← slightly higher: helps prevent overfitting
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/680 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
2.805 GB of memory reserved.


In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 680 | Num Epochs = 3 | Total steps = 255
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.248500
2,2.221300
3,2.175900
4,2.086400
5,2.025500
6,1.947100
7,1.848400
8,1.772000
9,1.640500
10,1.574600


Step,Training Loss
1,2.248500
2,2.221300
3,2.175900
4,2.086400
5,2.025500
6,1.947100
7,1.848400
8,1.772000
9,1.640500
10,1.574600


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

175.6403 seconds used for training.
2.93 minutes used for training.
Peak reserved memory = 3.287 GB.
Peak reserved memory for training = 0.482 GB.
Peak reserved memory % of max memory = 22.571 %.
Peak reserved memory for training % of max memory = 3.31 %.


In [9]:
# ════════════════════════════════════════════════════════════════════
# SAVE MODEL — Run this right after training so the work is never lost
# ════════════════════════════════════════════════════════════════════

# 1) Save the LoRA adapter locally in the Colab session
model.save_pretrained("logpulse_lora")
tokenizer.save_pretrained("logpulse_lora")
print("✅ LoRA adapter + tokenizer saved to ./logpulse_lora")

# 2) Zip it and download to your computer (most reliable backup)
import shutil
shutil.make_archive("logpulse_lora", "zip", "logpulse_lora")
print("✅ Zipped to logpulse_lora.zip")

from google.colab import files
files.download("logpulse_lora.zip")
print("⬇️  Downloading logpulse_lora.zip to your computer...")

✅ LoRA adapter + tokenizer saved to ./logpulse_lora
✅ Zipped to logpulse_lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading logpulse_lora.zip to your computer...


In [10]:
# ════════════════════════════════════════════════════════════════════
# Define SYSTEM_PROMPT (same one the model was trained on — must match exactly)
# Run this before the inference / evaluation cells.
# ════════════════════════════════════════════════════════════════════
SYSTEM_PROMPT = """You are a DevOps log classifier. Given a raw log line, output ONLY a valid JSON object with these exact fields:

{
  "category": one of ["Network", "Database", "Memory", "Security", "Application", "System"],
  "source": one of ["Docker", "Kubernetes", "Server", "Database", "OS", "Network_Device"],
  "severity": one of ["Warning", "Critical", "Fatal"],
  "summary": a short human-readable summary of the problem (max 15 words)
}

FIELD DEFINITIONS:
- "category" = WHAT kind of issue it is:
    • Network    → connections, timeouts, packets, sockets, DNS, ports
    • Database   → data storage, blocks, reads/writes, queries, replication, file system data ops (e.g. HDFS block storage)
    • Memory     → RAM, heap, OOM, leaks, allocation, cache eviction
    • Security   → auth, login attempts, permissions, intrusion, SSH, certificates
    • Application→ app-level logic, services, request handling, normal app operations
    • System     → OS/kernel/hardware level, daemons, processes, system services

- "source" = WHERE the log came from:
    • Docker        → container runtime logs
    • Kubernetes    → pods, deployments, kubelet, scheduler
    • Server        → application server processes (e.g. HDFS DataNode/NameNode)
    • Database      → dedicated DB engines
    • OS            → operating system / kernel
    • Network_Device→ routers, switches, firewalls

DISAMBIGUATION RULES (read carefully):
- HDFS / distributed-storage logs about BLOCKS, DataNode, NameNode storage operations → category "Database", source "Server".
- SSH / login / authentication logs → category "Security", source "OS".
- Kernel / hardware / daemon / kernel-panic logs → category "System", source "OS".
- Container crash / image pull / pod / OOMKilled logs → source "Kubernetes" or "Docker"; pick category by the actual problem (OOMKilled → "Memory", network timeout → "Network", crash loop → "Application").
- Be CONSISTENT: the same kind of log must always get the same category. Do not switch between "Application" and "System" for identical log types.

SEVERITY RULES:
- INFO / normal operational logs → "Warning"
- ERROR / failed operations / exceptions → "Critical"
- FATAL / crash / data loss / system down → "Fatal"

OUTPUT RULES:
- Output ONLY the JSON object. No markdown, no backticks, no explanation.
- Always pick the single best-fitting value for each field.

EXAMPLES:
Log: "081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049 terminating"
Output: {"category": "Database", "source": "Server", "severity": "Warning", "summary": "HDFS DataNode packet responder terminating for a storage block"}

Log: "Dec 10 06:55:46 LabSZ sshd[24200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2"
Output: {"category": "Security", "source": "OS", "severity": "Critical", "summary": "Failed SSH login attempt for invalid user from external IP"}

Log: "2026-05-20T10:14:22Z pod/api-server-7d9 OOMKilled: container exceeded memory limit 512Mi"
Output: {"category": "Memory", "source": "Kubernetes", "severity": "Critical", "summary": "Kubernetes pod killed for exceeding its memory limit"}

Log: "kernel: [12345.67] CPU0: Core temperature above threshold, cpu clock throttled"
Output: {"category": "System", "source": "OS", "severity": "Warning", "summary": "CPU core temperature exceeded threshold, clock throttled"}"""

print("✅ SYSTEM_PROMPT defined")

✅ SYSTEM_PROMPT defined


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [11]:
# ════════════════════════════════════════════════════════════════════
# INFERENCE TEST — Test the trained model on a NEW log (chat format)
# Replaces the Alpaca-style inference cell.
# ════════════════════════════════════════════════════════════════════
FastLanguageModel.for_inference(model)  # enable 2x faster inference

# A test log the model did NOT see during training
test_log = "Dec 10 09:12:33 LabSZ sshd[19541]: Failed password for root from 218.65.30.108 port 21105 ssh2"

# Build the conversation the same way we trained: system + user
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_log},
]

# Apply Qwen's chat template, add_generation_prompt=True so the model continues as assistant
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,    # signal the model to produce the assistant answer
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1,    # low temp = consistent, deterministic JSON
)

# Decode ONLY the newly generated part (skip the prompt)
generated = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)
print("LOG:", test_log)
print("\nMODEL OUTPUT:")
print(generated[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


LOG: Dec 10 09:12:33 LabSZ sshd[19541]: Failed password for root from 218.65.30.108 port 21105 ssh2

MODEL OUTPUT:
{"category": "Security", "source": "OS", "severity": "Critical", "summary": "Failed SSH password for user 'root' from remote IP."}


In [12]:
# ════════════════════════════════════════════════════════════════════
# HYBRID EVALUATION on test.jsonl (120 examples the model never saw)
#  - Rule-based accuracy for category / source / severity
#  - LLM-as-a-Judge (Gemini) for summary quality
# Requires: model, tokenizer, SYSTEM_PROMPT already defined in this notebook.
# Requires: GEMINI_API_KEY in Colab Secrets (for the judge part).
# ════════════════════════════════════════════════════════════════════
import json, re, time
from google import genai
from google.colab import userdata

FastLanguageModel.for_inference(model)

# ---- Load the held-out test set ----
test_data = [json.loads(l) for l in open("test.jsonl", encoding="utf-8")]
print(f"Loaded {len(test_data)} test examples\n")

# ---- Helper: run our trained model on one log, return parsed JSON (or None) ----
def predict(log_line):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": log_line},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        input_ids=inputs, max_new_tokens=128, use_cache=True, temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.batch_decode(out[:, inputs.shape[1]:], skip_special_tokens=True)[0]
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        return None  # invalid JSON

# ---- Run predictions on the whole test set ----
print("Running model on test set...")
predictions = []
for i, item in enumerate(test_data):
    # the user message is the raw log (messages[1] in our chat format)
    log_line = item["messages"][1]["content"]
    gold     = json.loads(item["messages"][2]["content"])  # the correct label
    pred     = predict(log_line)
    predictions.append({"log": log_line, "gold": gold, "pred": pred})
    if (i + 1) % 20 == 0:
        print(f"   {i+1}/{len(test_data)} done")

# ════════════════════════════════════════════════════════════════════
# PART 1 — Rule-based metrics
# ════════════════════════════════════════════════════════════════════
total       = len(predictions)
valid_json  = sum(1 for p in predictions if p["pred"] is not None)
cat_correct = src_correct = sev_correct = full_correct = 0

for p in predictions:
    if p["pred"] is None:
        continue
    g, pr = p["gold"], p["pred"]
    c = pr.get("category") == g["category"]
    s = pr.get("source")   == g["source"]
    v = pr.get("severity") == g["severity"]
    cat_correct  += c
    src_correct  += s
    sev_correct  += v
    full_correct += (c and s and v)

print("\n" + "="*50)
print("📊 RULE-BASED METRICS")
print("="*50)
print(f"Valid JSON rate : {valid_json}/{total}  ({valid_json/total*100:.1f}%)")
print(f"Category accuracy: {cat_correct}/{valid_json}  ({cat_correct/valid_json*100:.1f}%)")
print(f"Source accuracy  : {src_correct}/{valid_json}  ({src_correct/valid_json*100:.1f}%)")
print(f"Severity accuracy: {sev_correct}/{valid_json}  ({sev_correct/valid_json*100:.1f}%)")
print(f"Full 3-field match: {full_correct}/{valid_json}  ({full_correct/valid_json*100:.1f}%)")

Loaded 120 test examples

Running model on test set...
   20/120 done
   40/120 done
   60/120 done
   80/120 done
   100/120 done
   120/120 done

📊 RULE-BASED METRICS
Valid JSON rate : 120/120  (100.0%)
Category accuracy: 109/120  (90.8%)
Source accuracy  : 119/120  (99.2%)
Severity accuracy: 116/120  (96.7%)
Full 3-field match: 105/120  (87.5%)


TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [13]:
import re, time
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
JUDGE_MODEL = "gemini-2.5-flash"

JUDGE_PROMPT = """You are evaluating the quality of a one-line summary of a DevOps log.
Given the raw log and a candidate summary, rate the summary from 1 to 10 based on:
- Accuracy (does it correctly describe the log?)
- Clarity (is it clear and concise?)
Output ONLY a single integer from 1 to 10. No explanation."""

def judge_summary(log_line, summary):
    try:
        resp = client.models.generate_content(
            model=JUDGE_MODEL,
            contents=f"{JUDGE_PROMPT}\n\nLog: {log_line}\nSummary: {summary}",
        )
        nums = re.findall(r"\d+", resp.text)
        return int(nums[0]) if nums else None
    except Exception:
        return None

scores = []
for i, p in enumerate(predictions):   # predictions لسا محفوظ من قبل
    if p["pred"] is None or "summary" not in p["pred"]:
        continue
    s = judge_summary(p["log"], p["pred"]["summary"])
    if s is not None:
        scores.append(s)
    if (i + 1) % 20 == 0:
        print(f"   {i+1}/{len(predictions)} judged")
    time.sleep(0.3)

print(f"\nAverage summary score: {sum(scores)/len(scores):.2f}/10")
print(f"Scores >= 7: {sum(1 for s in scores if s>=7)}/{len(scores)} "
      f"({sum(1 for s in scores if s>=7)/len(scores)*100:.0f}%)")

   20/120 judged
   40/120 judged
   60/120 judged
   80/120 judged
   100/120 judged
   120/120 judged

Average summary score: 9.69/10
Scores >= 7: 117/117 (100%)


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

('qwen_lora/tokenizer_config.json',
 'qwen_lora/special_tokens_map.json',
 'qwen_lora/chat_template.jinja',
 'qwen_lora/vocab.json',
 'qwen_lora/merges.txt',
 'qwen_lora/added_tokens.json',
 'qwen_lora/tokenizer.json')

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>